# 03 · Automatización de archivos con pathlib

## Laboratorio de rutas y operaciones verificables

Trabajamos con archivos ficticios en una **carpeta temporal nueva**. No necesitas aportar documentos reales ni configurar rutas de tu ordenador. Las funciones de escritura y traslado se limitan a la raíz de la práctica.

El objetivo es comprender componentes, consultas, efectos y condiciones de cada operación. No implementamos todavía procesamiento masivo, backups ni el organizador completo.

**Versión alumnado:** guía completa y doce tareas pendientes.

## Objetivos y requisitos

Python 3.11 o superior, usando API disponible en 3.11. Biblioteca estándar: pathlib, tempfile, datetime, copy, shutil y json. Para abrir el notebook necesitas un editor compatible con Jupyter; los scripts no requieren Jupyter.

Al terminar podrás construir rutas portables, inventariar archivos y realizar operaciones individuales con previsualización y comprobaciones. El material toma como referencia Windows y Linux; ejecutar aquí no comprueba tu sistema Windows real.

## Cómo trabajar

Ejecuta en orden desde un kernel limpio. Antes de cada operación, predice qué debería existir después. Los doce ejercicios dependen de la guía, no de respuestas anteriores, y usan nombres independientes.

Activa cada bandera `completado_ejXX` después de resolverla. Una bandera False deja controles omitidos. Repetir una celda de creación puede producir una colisión esperada: no cambies el modo de apertura para ocultarla. Reiniciar la práctica completa genera un entorno nuevo.

## 1. Rutas antes que archivos

Un objeto Path representa una ubicación. Construirlo o consultar su nombre no crea nada. Las operaciones puras calculan componentes; otros métodos consultan el sistema o lo modifican. Los objetos son inmutables aunque el archivo representado cambie.

In [ ]:
"""Bloque 03: herramientas didácticas para un laboratorio temporal privado."""
from pathlib import Path, PureWindowsPath, PurePosixPath
from tempfile import TemporaryDirectory
from datetime import datetime, timezone
from copy import deepcopy
import shutil
import json

VERSION_LAB = "pathlib-aula-1.0"




In [ ]:
ruta_logica = Path("entrada") / "informe.final.txt"
print("Nombre:", ruta_logica.name)
print("Stem:", ruta_logica.stem)
print("Sufijo:", ruta_logica.suffix)
print("Sufijos:", ruta_logica.suffixes)
print("Padre:", ruta_logica.parent)


### Rutas de otro sistema sin acceso al disco

PureWindowsPath y PurePosixPath permiten analizar componentes sin abrir recursos. Un Path nativo de Linux no hace accesible una unidad E: de tu ordenador. Estas clases puras son útiles para aprender esa diferencia.

In [ ]:
windows_demo = PureWindowsPath("E:/observatorio/entrada/listado.csv")
posix_demo = PurePosixPath("/observatorio/entrada/listado.csv")
print("Unidad Windows:", windows_demo.drive)
print("Ancla Windows:", windows_demo.anchor)
print("Nombre:", windows_demo.name)
print("Partes POSIX:", posix_demo.parts)


### Contexto y composición

Path.cwd() consulta el directorio de trabajo. Una ruta relativa se interpreta desde él. En un script se puede utilizar __file__ para localizar su archivo, pero en un notebook normalmente no existe esa variable.

El operador / compone rutas. Un componente absoluto puede descartar la base; el operador no impide salir de una carpeta.

In [ ]:
print("Directorio de trabajo:", Path.cwd())
print("Relativa:", Path("entrada").is_absolute())
print("Componente absoluto POSIX:", PurePosixPath("/base") / "/otra")
propuesta = ruta_logica.with_name("informe_revisado.txt")
print("Calculada:", propuesta)
assert ruta_logica.name == "informe.final.txt"


## 2. Política conservadora de nombres

Validamos nombres simples de salida antes de combinarlos con una carpeta. La política rechaza separadores, caracteres de control, algunos caracteres no permitidos en Windows y nombres reservados frecuentes. No modifica mayúsculas ni acentos.

Esta política no es una lista exhaustiva de todos los nombres válidos o inválidos en cada sistema. El sistema de archivos puede imponer otras restricciones.

In [ ]:
def validar_nombre_simple(nombre):
    """Política conservadora de nombres para la práctica Windows/Linux."""
    if not isinstance(nombre, str) or not nombre or nombre in {".", ".."}:
        raise ValueError("Se requiere un nombre simple no vacío")
    if nombre != nombre.strip() or nombre.endswith("."):
        raise ValueError("No se admiten espacios extremos ni punto final")
    prohibidos = set('<>:"/\\|?*')
    if any(c in prohibidos or ord(c) < 32 for c in nombre):
        raise ValueError("Nombre con separadores o caracteres no admitidos")
    reservados = {"CON", "PRN", "AUX", "NUL"} | {
        f"{prefijo}{i}" for prefijo in ("COM", "LPT") for i in range(1, 10)
    }
    if nombre.split(".", 1)[0].upper() in reservados:
        raise ValueError("Nombre reservado en Windows")
    return nombre


In [ ]:
assert validar_nombre_simple("resumen_2025.txt") == "resumen_2025.txt"
for nombre_invalido in ["../fuera.txt", "CON.txt", "con:dos.txt", " final.txt"]:
    try:
        validar_nombre_simple(nombre_invalido)
    except ValueError as error:
        print(repr(nombre_invalido), "=>", error)


## 3. Control didáctico del alcance

La raíz debe existir y ser una carpeta privada de práctica. La función rechaza rutas fuera de ella, componentes .. y enlaces simbólicos en el recorrido.

**Límite:** comprobar y operar son pasos separados. El helper no es un sandbox frente a procesos hostiles que cambien rutas entre esos pasos. Se usa en un entorno temporal sin concurrencia.

In [ ]:
def asegurar_dentro(base, ruta, exigir_existencia=False):
    """Control didáctico de alcance, sin seguir enlaces ni aceptar '..'.

    La raíz debe ser una carpeta real, privada y sin modificaciones concurrentes.
    No constituye una frontera de seguridad frente a procesos hostiles.
    """
    raiz = Path(base).resolve(strict=True)
    if not raiz.is_dir():
        raise NotADirectoryError(raiz)
    candidata = Path(ruta)
    if not candidata.is_absolute():
        candidata = raiz / candidata
    if ".." in candidata.parts:
        raise ValueError("La política del laboratorio no admite '..'")
    try:
        relativa = candidata.relative_to(raiz)
    except ValueError as error:
        raise ValueError("Ruta fuera del laboratorio") from error
    paso = raiz
    for parte in relativa.parts:
        paso = paso / parte
        if paso.is_symlink():
            raise ValueError("La práctica no opera sobre enlaces simbólicos")
    resuelta = candidata.resolve(strict=exigir_existencia)
    if not resuelta.is_relative_to(raiz):
        raise ValueError("Destino resuelto fuera del laboratorio")
    return resuelta


## 4. Crear el laboratorio

La función crea una carpeta nueva con exist_ok=False, entrada, salida y siete archivos. Nunca vacía una carpeta anterior. Los contenidos son textos sintéticos en UTF-8. `archivo.tar.gz` solo ejemplifica un nombre compuesto: no contiene un comprimido real.

In [ ]:
def crear_entorno(base):
    """Crea datos nuevos; nunca reutiliza ni vacía una carpeta existente."""
    contenedor = Path(base).resolve(strict=True)
    raiz = contenedor / "laboratorio_pathlib"
    raiz.mkdir(exist_ok=False)
    entrada, salida = raiz / "entrada", raiz / "salida"
    entrada.mkdir()
    salida.mkdir()
    (entrada / "subcarpeta").mkdir()
    textos = {
        "listado_2024.csv": "id;importe;anio\nDEMO-001;100;2024\n",
        "listado_2025.CSV": "id;importe;anio\nDEMO-002;200;2025\n",
        "notas.txt": "Datos ficticios para aprender a manejar rutas.\n",
        "vacio.txt": "",
        "con espacios.txt": "La ruta puede contener espacios y acentos: revisión.\n",
        "subcarpeta/contexto.json": '{"territorio": "DEMO", "poblacion": 1000}\n',
        "archivo.tar.gz": "Ejemplo de nombre compuesto; no es un archivo comprimido.\n",
    }
    for relativa, texto in textos.items():
        with (entrada / relativa).open("x", encoding="utf-8", newline="\n") as f:
            f.write(texto)
    return {"raiz": raiz, "entrada": entrada, "salida": salida}


In [ ]:
temporal = TemporaryDirectory(prefix="curso_pathlib_")
entorno = crear_entorno(temporal.name)
raiz, entrada, salida = entorno["raiz"], entorno["entrada"], entorno["salida"]
print("Raíz temporal:", raiz)
print("Entrada:", entrada)
print("Salida:", salida)
assert entrada.is_dir() and salida.is_dir()


### Árbol inicial

![Entorno de la práctica](recursos/entorno.png)

Entrada tiene seis archivos directos y una subcarpeta. Dentro de esa subcarpeta hay un archivo más: siete archivos en total. La salida comienza vacía. La carpeta temporal cambiará de nombre en cada ejecución.

In [ ]:
for ruta in sorted(entrada.iterdir(), key=lambda p: p.name):
    print(ruta.name, "carpeta" if ruta.is_dir() else "archivo")
assert len(list(entrada.iterdir())) == 7
assert len([p for p in entrada.iterdir() if p.is_file()]) == 6


## 5. Selección explícita

Un patrón glob puede comportarse de forma distinta con mayúsculas según el entorno. Para obtener los dos CSV del laboratorio, filtramos archivos directos y comparamos suffix.lower(). Eso no modifica el nombre ni valida el formato interno.

In [ ]:
csv_directos = sorted(
    p for p in entrada.iterdir()
    if p.is_file() and p.suffix.lower() == ".csv"
)
print([p.name for p in csv_directos])
print("glob dependiente del entorno:", [p.name for p in entrada.glob("*.csv")])
assert [p.name for p in csv_directos] == ["listado_2024.csv", "listado_2025.CSV"]


### Recorrido recursivo

rglob recorre el árbol. Sus resultados deben filtrarse según la política del proceso. Aquí excluimos enlaces y conservamos archivos regulares. En un sistema que cambie durante el recorrido puede haber recursos que desaparezcan o permisos insuficientes.

In [ ]:
todos_iniciales = sorted(
    p for p in entrada.rglob("*") if not p.is_symlink() and p.is_file()
)
print([p.relative_to(entrada).as_posix() for p in todos_iniciales])
assert len(todos_iniciales) == 7


## 6. Inventario con metadatos

La ruta se expresa relativa a la raíz y con / para mostrarla. El tamaño lógico viene en bytes. La modificación se convierte a UTC. No interpretamos st_ctime como fecha de creación universal.

Si una consulta necesaria falla, se propaga el error: un resultado parcial no se presenta como inventario completo. El contrato está pensado para el pequeño entorno privado del laboratorio.

In [ ]:
def inventariar(base, carpeta):
    """Inventario explícito de archivos regulares, ordenado por ruta.

    Se recorren directorios con iterdir para no depender de errores que
    glob/rglob puedan omitir. Se excluyen enlaces y se propagan fallos.
    El árbol debe permanecer estable durante esta práctica sin concurrencia.
    """
    raiz = Path(base).resolve(strict=True)
    carpeta = asegurar_dentro(raiz, carpeta, exigir_existencia=True)
    if not carpeta.is_dir():
        raise NotADirectoryError(carpeta)
    filas, pendientes = [], [carpeta]
    while pendientes:
        actual = pendientes.pop()
        for ruta in sorted(actual.iterdir(), key=lambda p: p.as_posix()):
            if ruta.is_symlink():
                continue
            ruta = asegurar_dentro(raiz, ruta, exigir_existencia=True)
            if ruta.is_dir():
                pendientes.append(ruta)
            elif ruta.is_file():
                info = ruta.stat()
                filas.append({"ruta": ruta.relative_to(raiz).as_posix(),
                              "nombre": ruta.name, "extension": ruta.suffix.lower(),
                              "bytes": info.st_size,
                              "modificado_utc": datetime.fromtimestamp(
                                  info.st_mtime, tz=timezone.utc).isoformat()})
    return sorted(filas, key=lambda f: f["ruta"])


In [ ]:
inventario_inicial = inventariar(raiz, entrada)
for fila in inventario_inicial:
    print(fila["ruta"], fila["extension"], fila["bytes"])
total_bytes_inicial = sum(f["bytes"] for f in inventario_inicial)
print("Total bytes:", total_bytes_inicial)
print("MiB:", total_bytes_inicial / 1024**2)
assert len(inventario_inicial) == 7
assert sum(f["bytes"] == 0 for f in inventario_inicial) == 1


### Interpretación

![Distribución del inventario por sufijo final](recursos/inventario.png)

Dos CSV, tres TXT, un JSON y un nombre de sufijo GZ. Un archivo vacío sigue contando. Tamaños y fechas no demuestran identidad del contenido. Los nombres y fechas del entorno temporal no deben compararse literalmente entre ejecuciones distintas.

## 7. Leer texto y bytes

Los archivos se crearon con UTF-8 y esa codificación se declara al leer. read_text y read_bytes cargan todo el contenido. Para archivos grandes hay que analizar memoria y flujo de lectura; no se obtiene eficiencia por cambiar texto a bytes sin más.

In [ ]:
original_notas = entrada / "notas.txt"
texto_notas = original_notas.read_text(encoding="utf-8")
bytes_notas = original_notas.read_bytes()
print(texto_notas)
assert bytes_notas.decode("utf-8") == texto_notas
with (entrada / "listado_2024.csv").open("r", encoding="utf-8") as f:
    for linea in f:
        print(linea.rstrip("\n"))


## 8. Escribir una salida nueva

write_text reemplaza contenido si ya existe. Para este laboratorio usamos el modo x, que falla cuando el nombre existe. No basta con que el archivo estuviera vacío. La carpeta padre se crea de forma expresa antes de escribir.

Una escritura exclusiva no es una transacción completa: si falla a mitad, puede quedar una salida parcial. El bloque no implementa recuperación de lotes.

In [ ]:
def escribir_texto_nuevo(base, destino, texto):
    """Escritura exclusiva: falla si existe, incluso si está vacío."""
    if not isinstance(texto, str):
        raise TypeError("El contenido debe ser texto")
    destino = asegurar_dentro(base, destino)
    # La carpeta debe crearse expresamente antes de escribir.
    if not destino.parent.is_dir():
        raise FileNotFoundError(destino.parent)
    with destino.open("x", encoding="utf-8", newline="\n") as f:
        f.write(texto)
    return destino


In [ ]:
resumen_demo = escribir_texto_nuevo(
    raiz, salida / "resumen_demo.txt", "Inventario ficticio de aula.\n"
)
print(resumen_demo.read_text(encoding="utf-8"))
try:
    escribir_texto_nuevo(raiz, resumen_demo, "No debe reemplazar")
except FileExistsError:
    print("Colisión esperada: el primer contenido se conserva")
assert resumen_demo.read_text(encoding="utf-8") == "Inventario ficticio de aula.\n"


## 9. Copiar contenido sin reemplazar un destino existente

Path representa los extremos. shutil.copyfileobj realiza la copia por flujo. El destino se abre con xb para creación exclusiva. No se promete copiar todos los metadatos, propietarios ni permisos de acceso.

Una copia dentro del laboratorio es una copia de trabajo, no una estrategia de backup.

In [ ]:
def copiar_nuevo(base, origen, destino):
    """Copia el contenido con creación exclusiva del archivo de destino.

    No copia propietarios ni ACL ni promete preservación de metadatos.
    Si falla durante la escritura puede quedar una copia parcial para revisar.
    """
    origen = asegurar_dentro(base, origen, exigir_existencia=True)
    destino = asegurar_dentro(base, destino)
    if not origen.is_file():
        raise ValueError("El origen debe ser un archivo regular")
    if not destino.parent.is_dir():
        raise FileNotFoundError(destino.parent)
    with origen.open("rb") as fuente, destino.open("xb") as salida:
        shutil.copyfileobj(fuente, salida)
    return destino


In [ ]:
copia_demo = copiar_nuevo(raiz, original_notas, salida / "copia_notas.txt")
assert original_notas.exists() and copia_demo.exists()
assert original_notas.read_bytes() == copia_demo.read_bytes()
print("Contenido de copia contrastado con el original")


## 10. Previsualizar y ejecutar un traslado local

La función comprueba origen, destino, carpeta padre y colisiones. Solo admite archivos. Simula por defecto. Si se ejecuta, usa rename dentro del mismo sistema de archivos del laboratorio.

**Límite:** la comprobación de colisión no hace rename exclusivo frente a carreras en POSIX. Trabajamos con una raíz privada y un solo escritor. Los traslados entre unidades tienen otras condiciones y no se resuelven con este helper.

In [ ]:
def trasladar_local(base, origen, destino, simular=True):
    """Previsualiza o usa Path.rename en el laboratorio sin concurrencia.

    La comprobación previa de colisión NO hace rename exclusivo frente a
    carreras en POSIX. Solo se usa con datos temporales y un único escritor.
    No es una operación general entre unidades o sistemas de archivos.
    """
    if type(simular) is not bool:
        raise TypeError("simular debe ser un booleano")
    origen = asegurar_dentro(base, origen, exigir_existencia=True)
    destino = asegurar_dentro(base, destino)
    if not origen.is_file():
        raise ValueError("La práctica solo traslada archivos regulares")
    if origen == destino:
        raise ValueError("Origen y destino coinciden")
    if destino.exists() or destino.is_symlink():
        raise FileExistsError(destino)
    if not destino.parent.is_dir():
        raise FileNotFoundError(destino.parent)
    plan = {"origen": str(origen), "destino": str(destino),
            "ejecutado": not simular}
    if not simular:
        origen.rename(destino)
    return plan

def renombrar_local(base, origen, nuevo_nombre, simular=True):
    validar_nombre_simple(nuevo_nombre)
    origen = asegurar_dentro(base, origen, exigir_existencia=True)
    return trasladar_local(base, origen, origen.with_name(nuevo_nombre), simular)


In [ ]:
plan_demo = renombrar_local(raiz, copia_demo, "notas_revisadas.txt")
print(plan_demo)
assert plan_demo["ejecutado"] is False
assert copia_demo.exists()
assert not (salida / "notas_revisadas.txt").exists()


In [ ]:
renombrar_local(raiz, copia_demo, "notas_revisadas.txt", simular=False)
renombrada_demo = salida / "notas_revisadas.txt"
assert renombrada_demo.exists() and not copia_demo.exists()
carpeta_archivadas = salida / "archivadas"
carpeta_archivadas.mkdir()
trasladar_local(raiz, renombrada_demo, carpeta_archivadas / renombrada_demo.name,
                simular=False)
assert not renombrada_demo.exists()
assert (carpeta_archivadas / "notas_revisadas.txt").read_bytes() == bytes_notas


### La ruta antigua no se actualiza

Después de mover el recurso, el objeto que representaba la ubicación anterior sigue representando ese nombre antiguo. Guarda o construye la nueva ruta y comprueba el estado allí.

Las entradas originales permanecen intactas. El renombrado y el movimiento se han hecho sobre una copia de trabajo.

## 11. Borrar solo un ejemplo dedicado

unlink no envía a la papelera. Esta demostración crea un archivo de descarte, muestra el plan y solo después elimina ese archivo. No borra carpetas ni entradas originales. rmdir se utiliza únicamente para una carpeta vacía que acabamos de crear.

In [ ]:
def borrar_demo(base, ruta, simular=True):
    """Solo un archivo del laboratorio. Sin borrado recursivo."""
    if type(simular) is not bool:
        raise TypeError("simular debe ser un booleano")
    ruta = asegurar_dentro(base, ruta, exigir_existencia=True)
    if not ruta.is_file():
        raise ValueError("Solo se admite un archivo regular")
    if not simular:
        ruta.unlink()
    return {"ruta": str(ruta), "eliminado": not simular}


In [ ]:
descarte_demo = escribir_texto_nuevo(raiz, salida / "descarte_demo.txt", "Descartar")
print(borrar_demo(raiz, descarte_demo))
assert descarte_demo.exists()
borrar_demo(raiz, descarte_demo, simular=False)
assert not descarte_demo.exists()
vacia_demo = salida / "vacia_demo"
vacia_demo.mkdir()
vacia_demo.rmdir()
assert not vacia_demo.exists()
assert original_notas.read_bytes() == bytes_notas


## 12. Contrastar el estado final de la guía

La entrada debe conservar sus siete archivos. Las fechas de modificación de archivos nuevos y el nombre temporal varían entre sesiones. Para comprobar la conservación comparamos contenido y nombres dentro de esta ejecución.

In [ ]:
inventario_entrada_final = inventariar(raiz, entrada)
assert [(f["ruta"],f["bytes"]) for f in inventario_inicial] == [
    (f["ruta"],f["bytes"]) for f in inventario_entrada_final
]
assert original_notas.read_bytes() == bytes_notas
print("Entrada original conservada; salidas:")
for f in inventariar(raiz, salida):
    print(f["ruta"])


## Ejercicios

Cada tarea usa nombres propios en salida para no depender de otras respuestas. La guía conserva los archivos de entrada. Completa la solución y activa su bandera para ejecutar controles. No apuntes las funciones a tus carpetas reales.

Si necesitas repetir todo, reinicia el kernel y ejecuta desde el comienzo: se creará un contenedor nuevo. Si repites una sola celda de creación, puede aparecer una colisión intencionada.

## Ejercicio 01 · Leer componentes

Define `ruta_ej1` como Path("entrada") / "informe.final.txt" y `componentes_ej1` con nombre, stem y extension. No consultes ni crees archivos.

**Pista:** Las propiedades operan sobre la ruta aunque el recurso no exista.

In [ ]:
ruta_ej1 = None  # TODO
componentes_ej1 = {}

In [ ]:
completado_ej01 = False
if completado_ej01:
    assert componentes_ej1=={"nombre":"informe.final.txt","stem":"informe.final","extension":".txt"}
    assert ruta_ej1.parent==Path("entrada")
    print("Ejercicio 01: controles superados")
else:
    print("Ejercicio 01: pendiente; controles omitidos")

**Justifica:** describe las precondiciones, el efecto esperado y una limitación de la operación.

## Ejercicio 02 · Rutas Windows sin disco Windows

Crea `win_ej2` con PureWindowsPath("E:/observatorio/entrada/datos.csv"). Guarda `unidad_ej2` y `nombre_ej2`. Explica en `limite_ej2` por qué eso no permite abrir la unidad desde Linux.

**Pista:** Es una clase de operaciones puras.

In [ ]:
win_ej2 = None  # TODO
unidad_ej2 = None
nombre_ej2 = None
limite_ej2 = ""

In [ ]:
completado_ej02 = False
if completado_ej02:
    assert unidad_ej2=="E:"
    assert nombre_ej2=="datos.csv"
    assert len(limite_ej2)>35
    print("Ejercicio 02: controles superados")
else:
    print("Ejercicio 02: pendiente; controles omitidos")

**Justifica:** describe las precondiciones, el efecto esperado y una limitación de la operación.

## Ejercicio 03 · Cambiar una ruta sin cambiar el archivo

A partir de `entrada / "archivo.tar.gz"`, calcula `propuesta_ej3` con sufijo .txt. Guarda `nombre_ej3`. Comprueba que el origen sigue existiendo y la ruta propuesta no se ha creado.

**Pista:** with_suffix cambia solo el sufijo final.

In [ ]:
origen_ej3 = entrada / "archivo.tar.gz"
propuesta_ej3 = None  # TODO
nombre_ej3 = None

In [ ]:
completado_ej03 = False
if completado_ej03:
    assert nombre_ej3=="archivo.tar.txt"
    assert origen_ej3.exists()
    assert not propuesta_ej3.exists()
    print("Ejercicio 03: controles superados")
else:
    print("Ejercicio 03: pendiente; controles omitidos")

**Justifica:** describe las precondiciones, el efecto esperado y una limitación de la operación.

## Ejercicio 04 · Hijos directos y archivos de todo el árbol

Define `n_hijos_ej4`, `n_archivos_directos_ej4` y `n_archivos_arbol_ej4`. Usa iterdir para los dos primeros y rglob con filtro de archivo para el tercero.

**Pista:** Una subcarpeta es un hijo, pero no un archivo regular.

In [ ]:
n_hijos_ej4 = None  # TODO
n_archivos_directos_ej4 = None
n_archivos_arbol_ej4 = None

In [ ]:
completado_ej04 = False
if completado_ej04:
    assert n_hijos_ej4==7
    assert n_archivos_directos_ej4==6
    assert n_archivos_arbol_ej4==7
    print("Ejercicio 04: controles superados")
else:
    print("Ejercicio 04: pendiente; controles omitidos")

**Justifica:** describe las precondiciones, el efecto esperado y una limitación de la operación.

## Ejercicio 05 · Seleccionar CSV con criterio explícito

Obtén `csvs_ej5` como lista ordenada de nombres de archivos directos cuya extensión, comparada en minúsculas, sea .csv. Conserva el nombre real.

**Pista:** No confíes en el comportamiento de mayúsculas de glob en todos los sistemas.

In [ ]:
csvs_ej5 = []  # TODO

In [ ]:
completado_ej05 = False
if completado_ej05:
    assert csvs_ej5==["listado_2024.csv","listado_2025.CSV"]
    print("Ejercicio 05: controles superados")
else:
    print("Ejercicio 05: pendiente; controles omitidos")

**Justifica:** describe las precondiciones, el efecto esperado y una limitación de la operación.

## Ejercicio 06 · Inventariar y calcular unidades

Obtén `inventario_ej6`, `bytes_ej6` y `mib_ej6` de la entrada. Usa inventariar y calcula el tamaño total. Comprueba que el archivo vacío se conserva.

**Pista:** MiB divide bytes entre 1024**2, no entre 1000.

In [ ]:
inventario_ej6 = inventariar(raiz,entrada)
bytes_ej6 = None  # TODO
mib_ej6 = None

In [ ]:
completado_ej06 = False
if completado_ej06:
    assert len(inventario_ej6)==7
    assert bytes_ej6==sum(p.stat().st_size for p in entrada.rglob("*") if p.is_file())
    assert mib_ej6==bytes_ej6/1024**2
    assert any(f["nombre"]=="vacio.txt" and f["bytes"]==0 for f in inventario_ej6)
    print("Ejercicio 06: controles superados")
else:
    print("Ejercicio 06: pendiente; controles omitidos")

**Justifica:** describe las precondiciones, el efecto esperado y una limitación de la operación.

## Ejercicio 07 · Texto UTF-8 y nueva salida

Lee notas.txt con UTF-8 en `texto_ej7`. Crea salida/ej07.txt con el contenido "Revisión de aula: ñ y á.\n" usando escribir_texto_nuevo. Guarda la ruta en `escrito_ej7`.

**Pista:** El padre salida ya existe. La creación debe ser exclusiva.

In [ ]:
texto_ej7 = None  # TODO
escrito_ej7 = None

In [ ]:
completado_ej07 = False
if completado_ej07:
    assert texto_ej7==texto_notas
    assert escrito_ej7.read_text(encoding="utf-8")=="Revisión de aula: ñ y á.\n"
    assert escrito_ej7.read_bytes().decode("utf-8")=="Revisión de aula: ñ y á.\n"
    print("Ejercicio 07: controles superados")
else:
    print("Ejercicio 07: pendiente; controles omitidos")

**Justifica:** describe las precondiciones, el efecto esperado y una limitación de la operación.

## Ejercicio 08 · Una copia no elimina el origen

Copia notas.txt a salida/ej08_copia.txt con copiar_nuevo y guarda `copia_ej8`. Contrasta los bytes y la presencia del origen.

**Pista:** Para estos archivos pequeños, leer bytes permite comparar el contenido sin analizarlo.

In [ ]:
copia_ej8 = None  # TODO

In [ ]:
completado_ej08 = False
if completado_ej08:
    assert copia_ej8.exists()
    assert (entrada / "notas.txt").exists()
    assert copia_ej8.read_bytes()==(entrada / "notas.txt").read_bytes()
    print("Ejercicio 08: controles superados")
else:
    print("Ejercicio 08: pendiente; controles omitidos")

**Justifica:** describe las precondiciones, el efecto esperado y una limitación de la operación.

## Ejercicio 09 · Previsualización sin cambio

Se proporciona un archivo nuevo para este ejercicio. Obtén `plan_ej9` proponiendo renombrarlo a ej09_nuevo.txt. Mantén simular=True y verifica la ausencia de efecto.

**Pista:** El helper simula por defecto, pero puedes escribir el parámetro para hacer visible la intención.

In [ ]:
origen_ej9 = escribir_texto_nuevo(raiz,salida / "ej09_origen.txt","Ejercicio nueve")
plan_ej9 = None  # TODO

In [ ]:
completado_ej09 = False
if completado_ej09:
    assert plan_ej9["ejecutado"] is False
    assert origen_ej9.exists()
    assert not (salida / "ej09_nuevo.txt").exists()
    print("Ejercicio 09: controles superados")
else:
    print("Ejercicio 09: pendiente; controles omitidos")

**Justifica:** describe las precondiciones, el efecto esperado y una limitación de la operación.

## Ejercicio 10 · Traslado real dentro del entorno

Se proporciona un origen independiente. Crea la carpeta salida/ej10_destino y mueve el archivo a resultado.txt con trasladar_local y simular=False. Guarda `movido_ej10` como Path al destino.

**Pista:** Primero crea el padre. El objeto del origen seguirá representando la ruta antigua.

In [ ]:
origen_ej10 = escribir_texto_nuevo(raiz,salida / "ej10_origen.txt","Contenido diez")
carpeta_ej10 = salida / "ej10_destino"
# TODO: crea carpeta, mueve y define la nueva ruta
movido_ej10 = None

In [ ]:
completado_ej10 = False
if completado_ej10:
    assert not origen_ej10.exists()
    assert movido_ej10.is_file()
    assert movido_ej10.read_text(encoding="utf-8")=="Contenido diez"
    print("Ejercicio 10: controles superados")
else:
    print("Ejercicio 10: pendiente; controles omitidos")

**Justifica:** describe las precondiciones, el efecto esperado y una limitación de la operación.

## Ejercicio 11 · Colisión y alcance

Se crea un destino ocupado. Captura FileExistsError al intentar copiar notas.txt sobre él y guarda `colision_ej11=True`. Intenta escribir en `raiz.parent / "fuera.txt"` y captura ValueError con `alcance_ej11=True`.

**Pista:** No uses un except genérico. El primer contenido debe permanecer intacto y no debe aparecer un archivo fuera de la raíz.

In [ ]:
ocupado_ej11 = escribir_texto_nuevo(raiz,salida / "ej11_ocupado.txt","Conservar")
fuera_ej11 = raiz.parent / "fuera.txt"
colision_ej11 = False
alcance_ej11 = False
# TODO: dos intentos con sus excepciones esperadas

In [ ]:
completado_ej11 = False
if completado_ej11:
    assert colision_ej11 and alcance_ej11
    assert ocupado_ej11.read_text(encoding="utf-8")=="Conservar"
    assert not fuera_ej11.exists()
    print("Ejercicio 11: controles superados")
else:
    print("Ejercicio 11: pendiente; controles omitidos")

**Justifica:** describe las precondiciones, el efecto esperado y una limitación de la operación.

## Ejercicio 12 · Eliminar solo un archivo creado para descartar

Se crea un archivo de descarte. Obtén `plan_ej12` con simulación, verifica que aún existe y elimina luego solo ese archivo con simular=False. Añade `limite_ej12` sobre unlink y papelera.

**Pista:** No borres carpetas ni el archivo original de entrada.

In [ ]:
descarte_ej12 = escribir_texto_nuevo(raiz,salida / "ej12_descartar.txt","Solo descartar este ejemplo")
plan_ej12 = None  # TODO
limite_ej12 = ""
# TODO: simula, comprueba y elimina este ejemplo

In [ ]:
completado_ej12 = False
if completado_ej12:
    assert plan_ej12["eliminado"] is False
    assert not descarte_ej12.exists()
    assert (entrada / "notas.txt").read_bytes()==bytes_notas
    assert len(limite_ej12)>40
    print("Ejercicio 12: controles superados")
else:
    print("Ejercicio 12: pendiente; controles omitidos")

**Justifica:** describe las precondiciones, el efecto esperado y una limitación de la operación.

## Cierre y retirada del entorno

La raíz temporal no es un lugar de conservación. Si activas la siguiente bandera, se elimina únicamente el contenedor creado al comenzar. No hagas la retirada hasta terminar de inspeccionar la práctica.

Al reiniciar o cerrar el kernel, el objeto temporal puede retirar sus archivos. Para repetir la sesión completa crea un entorno nuevo desde el inicio.

In [ ]:
retirar_entorno_ahora = False
if retirar_entorno_ahora:
    temporal.cleanup()
    print("Entorno temporal propio retirado")
else:
    print("El entorno sigue disponible durante esta sesión:", raiz)


## Entrega y evaluación

Entrega las doce tareas completadas y sus controles activados, con razonamiento sobre rutas, efectos, colisiones y alcance. No incluyas datos ni carpetas reales. Cinco dimensiones, dos puntos cada una: componentes y contexto, exploración, lectura/escritura, operaciones con evidencia y explicación de límites.

## Fuentes

- https://docs.python.org/3.11/library/pathlib.html
- https://docs.python.org/3.11/library/shutil.html
- https://docs.python.org/3.11/library/tempfile.html
- https://docs.python.org/3.11/library/functions.html#open

Los helpers de simulación y control son originales y didácticos. Referencias de Python 3.11 revisadas el 03/11/2026.